# Explorador do banco **Chinook** (PostgreSQL)

Este notebook conecta-se ao banco **`chinook` já existente no PostgreSQL** (ou seja, **não** lê os dados do arquivo `Chinook_PostgreSql.sql`) e gera, para **todas as tabelas**:

- o que cada tabela **representa**;
- a **estrutura** (colunas, tipos, obrigatoriedade);
- a **contagem de registros** e de valores nulos;
- os **valores únicos** de cada coluna;
- uma **amostra** de linhas.

## Como usar
1. Ajuste as credenciais na célula de **Configuração** (host, porta, banco, usuário e senha).
2. Execute tudo com **Run All** (ou `Shift+Enter` célula a célula, em ordem).


In [ ]:
# Verifica as dependências necessárias.
# Se aparecer alguma faltando, execute:  %pip install pandas sqlalchemy psycopg2-binary
import importlib.util

faltando = [m for m in ("pandas", "sqlalchemy", "psycopg2") if importlib.util.find_spec(m) is None]
if faltando:
    print("Faltam dependências:", ", ".join(faltando))
    print("Instale com:  %pip install pandas sqlalchemy psycopg2-binary")
else:
    print("Dependências OK.")


In [ ]:
import pandas as pd
from urllib.parse import quote_plus
from sqlalchemy import create_engine, text
from IPython.display import display, Markdown

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

# ===========================================================================
# CONFIGURAÇÃO DA CONEXÃO  (ajuste conforme o seu PostgreSQL)
# ===========================================================================
DB_CONFIG = {
    "host":     "localhost",   # ex.: "localhost" ou "127.0.0.1"
    "port":     5432,
    "database": "chinook",     # banco já criado/populado no PostgreSQL
    "user":     "postgres",    # usuário do PostgreSQL
    "password": "",            # <-- informe a senha do usuário
}


In [ ]:
# Cria a conexão e testa
url = (
    f"postgresql+psycopg2://{quote_plus(DB_CONFIG['user'])}:"
    f"{quote_plus(DB_CONFIG['password'])}@"
    f"{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
)
engine = create_engine(url)

with engine.connect() as conn:
    banco_atual = conn.execute(text("SELECT current_database()")).scalar()
    versao = conn.execute(text("SELECT version()")).scalar()

print("Conectado ao banco:", banco_atual)
print(versao)


## 1. O que cada tabela representa

Dicionário com a finalidade de cada tabela do Chinook.


In [ ]:
# Descrição de cada tabela do Chinook.
DESCRICOES = {
    "album":          "Álbuns musicais. Cada álbum pertence a um artista (artist_id).",
    "artist":         "Artistas / bandas que possuem álbuns cadastrados.",
    "customer":       "Clientes da loja, com dados de contato. support_rep_id indica o funcionário de suporte.",
    "employee":       "Funcionários da empresa. reports_to aponta para o gerente (auto-relacionamento).",
    "genre":          "Gêneros musicais (Rock, Jazz, Classical, etc.).",
    "invoice":        "Notas fiscais / vendas. Cada venda pertence a um cliente (customer_id).",
    "invoice_line":   "Itens de cada nota fiscal; liga uma venda (invoice) a uma faixa (track).",
    "media_type":     "Tipos de mídia dos arquivos das faixas (MPEG, AAC, etc.).",
    "playlist":       "Playlists criadas pelos usuários.",
    "playlist_track": "Tabela de associação N:N entre playlists e faixas.",
    "track":          "Faixas musicais; cada faixa pertence a um álbum e tem gênero, tipo de mídia, duração e preço.",
}

def descrever(tabela):
    return DESCRICOES.get(tabela, "Tabela sem descrição cadastrada.")


## 2. Lista das tabelas do banco


In [ ]:
tabelas = pd.read_sql(text("""
    SELECT table_schema AS schema, table_name AS tabela
    FROM information_schema.tables
    WHERE table_type = 'BASE TABLE'
      AND table_schema NOT IN ('pg_catalog', 'information_schema')
    ORDER BY table_schema, table_name
"""), engine)

display(tabelas)

# Lista de tuplas (schema, tabela) que será usada nas próximas etapas
alvos = list(tabelas.itertuples(index=False, name=None))
print(f"{len(alvos)} tabela(s) encontrada(s).")


## 3. Análise de cada tabela

A função abaixo coleta, para uma tabela: colunas/tipos, total de registros, quantidade de nulos, quantidade de valores distintos e os **valores únicos** de cada coluna (limitados a um máximo configurável).


In [ ]:
def analisar_tabela(schema, tabela, max_valores=25, max_amostra=25):
    """Retorna um dicionário com estrutura, contagens, valores únicos e amostra."""
    q = engine.dialect.identifier_preparer.quote
    tabela_sql = f"{q(schema)}.{q(tabela)}"

    # --- Estrutura das colunas ------------------------------------------------
    colunas = pd.read_sql(text("""
        SELECT column_name  AS coluna,
               data_type    AS tipo,
               is_nullable  AS aceita_nulo,
               character_maximum_length AS tamanho
        FROM information_schema.columns
        WHERE table_schema = :s AND table_name = :t
        ORDER BY ordinal_position
    """), engine, params={"s": schema, "t": tabela})

    if colunas.empty:
        return None

    nomes = colunas["coluna"].tolist()

    # --- Total, distintos e nulos em uma única consulta -----------------------
    partes = []
    for i, c in enumerate(nomes):
        partes.append(f'COUNT(DISTINCT {q(c)}) AS "d_{i}"')
        partes.append(f'COUNT(*) FILTER (WHERE {q(c)} IS NULL) AS "n_{i}"')
    sql_resumo = f'SELECT COUNT(*) AS total, {", ".join(partes)} FROM {tabela_sql}'

    resumo = pd.read_sql(text(sql_resumo), engine)
    total = int(resumo["total"].iloc[0])

    colunas["distintos"] = [int(resumo[f"d_{i}"].iloc[0]) for i in range(len(nomes))]
    colunas["nulos"]     = [int(resumo[f"n_{i}"].iloc[0]) for i in range(len(nomes))]

    # --- Valores únicos (limitados) ------------------------------------------
    valores_unicos = {}
    for c in nomes:
        vals = pd.read_sql(
            text(f'SELECT DISTINCT {q(c)} AS v FROM {tabela_sql} '
                 f'WHERE {q(c)} IS NOT NULL ORDER BY 1'),
            engine,
        )["v"].tolist()
        valores_unicos[c] = vals

    # --- Amostra --------------------------------------------------------------
    amostra = pd.read_sql(text(f'SELECT * FROM {tabela_sql} LIMIT {int(max_amostra)}'), engine)

    return {
        "schema": schema,
        "tabela": tabela,
        "total": total,
        "colunas": colunas,
        "valores_unicos": valores_unicos,
        "amostra": amostra,
        "max_valores": max_valores,
    }


In [ ]:
# Executa a análise para todas as tabelas
relatorio = {}
erros = {}

for schema, tabela in alvos:
    try:
        relatorio[(schema, tabela)] = analisar_tabela(schema, tabela)
    except Exception as e:
        erros[(schema, tabela)] = str(e)

print(f"Analisadas: {len(relatorio)} tabela(s).")
if erros:
    print("Falhas:", erros)


## 4. Resumo geral


In [ ]:
linhas = []
for (schema, tabela), info in relatorio.items():
    n_cols_unicas = sum(1 for v in info["valores_unicos"].values() if len(v) <= info["max_valores"])
    linhas.append({
        "schema": schema,
        "tabela": tabela,
        "descrição": descrever(tabela),
        "registros": info["total"],
        "colunas": len(info["colunas"]),
        "colunas_analisadas": n_cols_unicas,
    })

resumo = pd.DataFrame(linhas).sort_values(["schema", "tabela"]).reset_index(drop=True)
display(resumo)


## 5. Detalhes por tabela

Colunas, contagens e **valores únicos** de cada tabela. Por padrão são exibidos até **25 valores** por coluna; ajuste `MAX_VALORES` abaixo se quiser mais (ou menos).


In [ ]:
MAX_VALORES = 25   # máximo de valores únicos exibidos por coluna

def mostrar_tabela(schema, tabela):
    info = relatorio[(schema, tabela)]
    display(Markdown(f"## `{schema}.{tabela}`"))
    display(Markdown(f"**O que representa:** {descrever(tabela)}"))
    display(Markdown(f"**Registros:** {info['total']}"))

    display(Markdown("**Colunas:**"))
    display(info["colunas"])

    display(Markdown("**Valores únicos por coluna:**"))
    for col, vals in info["valores_unicos"].items():
        d = len(vals)
        if d == 0:
            display(Markdown(f"- **{col}**: (sem valores)"))
        elif d <= MAX_VALORES:
            amostra = ", ".join(map(str, vals))
            display(Markdown(f"- **{col}** — {d} valor(es): {amostra}"))
        else:
            amostra = ", ".join(map(str, vals[:MAX_VALORES]))
            display(Markdown(f"- **{col}** — {d} valores (mostrando {MAX_VALORES}): {amostra} ..."))

    display(Markdown("**Amostra de linhas:**"))
    display(info["amostra"])

for schema, tabela in relatorio:
    mostrar_tabela(schema, tabela)


## 6. Relacionamentos entre as tabelas

As chaves estrangeiras conectam as tabelas assim:

- `album.artist_id` → `artist.artist_id`
- `track.album_id` → `album.album_id`
- `track.genre_id` → `genre.genre_id`
- `track.media_type_id` → `media_type.media_type_id`
- `customer.support_rep_id` → `employee.employee_id`
- `employee.reports_to` → `employee.employee_id` (auto-relacionamento)
- `invoice.customer_id` → `customer.customer_id`
- `invoice_line.invoice_id` → `invoice.invoice_id`
- `invoice_line.track_id` → `track.track_id`
- `playlist_track.playlist_id` → `playlist.playlist_id`
- `playlist_track.track_id` → `track.track_id`

Em resumo: **artistas** têm **álbuns**, que têm **faixas**; faixas têm **gênero** e **tipo de mídia**. **Clientes** fazem **vendas** (`invoice`) compostas por **itens** (`invoice_line`) que apontam para **faixas**. **Playlists** agrupam faixas via `playlist_track`, e **funcionários** atendem clientes e se reportam a um gerente.


## 7. Verificação: todos os dados foram carregados?

Compara a contagem real de cada tabela com o total esperado do Chinook. Se alguma tabela aparecer **VAZIA** ou **PARCIAL**, o carregamento não terminou (geralmente porque houve um erro no meio do script).


In [ ]:
# Contagens esperadas (script oficial Chinook 1.4.5)
ESPERADO = {
    "album": 347, "artist": 275, "customer": 59, "employee": 8, "genre": 25,
    "invoice": 412, "invoice_line": 2240, "media_type": 5, "playlist": 18,
    "playlist_track": 8715, "track": 3503,
}

linhas = []
for tabela, esperado in ESPERADO.items():
    real = int(pd.read_sql(text(f'SELECT COUNT(*) AS n FROM "{tabela}"'), engine)["n"].iloc[0])
    status = "OK" if real == esperado else ("VAZIA" if real == 0 else "PARCIAL")
    linhas.append({"tabela": tabela, "esperado": esperado, "no_banco": real,
                   "faltando": esperado - real, "status": status})

verificacao = pd.DataFrame(linhas).sort_values("tabela").reset_index(drop=True)
display(verificacao)

if (verificacao["status"] != "OK").any():
    print("ATENÇÃO: existem tabelas vazias/parciais -> o script nao foi executado por completo nesta base.")
else:
    print("Todos os dados foram carregados corretamente.")


In [ ]:
# Integridade referencial: procura chaves estrangeiras "orfas"
fk_checks = [
    ("album.artist_id -> artist", "album", "artist_id", "artist", "artist_id"),
    ("track.album_id -> album", "track", "album_id", "album", "album_id"),
    ("track.genre_id -> genre", "track", "genre_id", "genre", "genre_id"),
    ("track.media_type_id -> media_type", "track", "media_type_id", "media_type", "media_type_id"),
    ("customer.support_rep_id -> employee", "customer", "support_rep_id", "employee", "employee_id"),
    ("invoice.customer_id -> customer", "invoice", "customer_id", "customer", "customer_id"),
    ("invoice_line.invoice_id -> invoice", "invoice_line", "invoice_id", "invoice", "invoice_id"),
    ("invoice_line.track_id -> track", "invoice_line", "track_id", "track", "track_id"),
    ("playlist_track.playlist_id -> playlist", "playlist_track", "playlist_id", "playlist", "playlist_id"),
    ("playlist_track.track_id -> track", "playlist_track", "track_id", "track", "track_id"),
]

resultado_fk = []
for nome, ct, cc, pt, pc in fk_checks:
    sql = (f'SELECT COUNT(*) AS n FROM "{ct}" c '
           f'LEFT JOIN "{pt}" p ON c."{cc}" = p."{pc}" '
           f'WHERE c."{cc}" IS NOT NULL AND p."{pc}" IS NULL')
    orfaos = int(pd.read_sql(text(sql), engine)["n"].iloc[0])
    resultado_fk.append({"relacionamento": nome, "orfaos": orfaos,
                         "status": "OK" if orfaos == 0 else "FALHA"})

display(pd.DataFrame(resultado_fk))


## 8. Exportar o relatório (opcional)

Salva o resumo e o dicionário de dados (colunas + valores únicos) em CSV.


In [ ]:
import os

pasta = "relatorio_chinook"
os.makedirs(pasta, exist_ok=True)

resumo.to_csv(os.path.join(pasta, "resumo_tabelas.csv"), index=False)

for (schema, tabela), info in relatorio.items():
    info["colunas"].to_csv(os.path.join(pasta, f"{tabela}_colunas.csv"), index=False)
    with open(os.path.join(pasta, f"{tabela}_valores_unicos.txt"), "w", encoding="utf-8") as f:
        f.write(f"{tabela}: {descrever(tabela)}\n\n")
        for col, vals in info["valores_unicos"].items():
            f.write(f"{col} ({len(vals)}): {vals}\n")

print("Arquivos gerados em:", os.path.abspath(pasta))
